# DenseNet121

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_loader import get_dataloaders

## Setup

In [ ]:
from data_loader import get_dataloaders, DataLoaderConfig

# # Initialize configuration
# config = DataLoaderConfig(
#     batch_size=32,
#     image_size=224,       # Default DenseNet input size
#     num_workers=0         # Keeping at 0 for Jupyter on Windows
# )

# # Get dataloaders for train, valid, and test sets
dataloaders, class_names = get_dataloaders()

print(f"Classes: {class_names}")
print(f"Train batches: {len(dataloaders['train'])}")
print(f"Valid batches: {len(dataloaders['valid'])}")
print(f"Test batches: {len(dataloaders['test'])}")

## DenseNet121 Implementation

In [ ]:
# BN, ReLU, then conv 1x1 & 3x3, concatenated with input
class Bottleneck(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super(Bottleneck, self).__init__()
        
        inner_channels = 4 * growth_rate # 4k
        
        self.bottleneck = nn.Sequential(
            # 1x1 convolution 
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, inner_channels, kernel_size=1, bias=False),
            # 3x3 convolution
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(inner_channels, growth_rate, kernel_size=3, padding=1, bias=False)
        )

    def forward(self, x):
        return torch.cat([x, self.bottleneck(x)], 1) # concatenate input with output of bottleneck

# Dense block has multiple bottleneck layers, each one increasing the number of channels by the growth rate
class DenseBlock(nn.Module):
    def __init__(self, in_channels, growth_rate, num_layers):
        super(DenseBlock, self).__init__()
        
        layers = []
        
        for i in range(num_layers):
            size = in_channels + i * growth_rate
            layers.append(Bottleneck(size, growth_rate))
            
        self.dense_block = nn.Sequential(*layers)
            
    def forward(self, x):
        return self.dense_block(x)

# 1x1 conv then 2x2 average pool stride 2
class Transition(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Transition, self).__init__()
        
        self.transition = nn.Sequential(
            # 1x1 conv
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            # 2x2 average pool stride 2
            nn.AvgPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        return self.transition(x)

class DenseNet121(nn.Module):
    def __init__(self, num_classes=4, growth_rate=32):
        super(DenseNet121, self).__init__()
        
        self.growth_rate = growth_rate
        
        # each block has 1x1 conv followed by 3x3 conv times the number of layers in that block
        num_blocks = [6, 12, 24, 16] # number of layers in each dense block for DenseNet-121
        
        inner_channels = 2 * growth_rate

        # initial Convolution before the dense blocks
        self.features = nn.Sequential(
            # initial 7x7 conv stride 2 padding 3
            nn.Conv2d(3, inner_channels, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(inner_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # dense blocks and transition layers
        for i, num_layers in enumerate(num_blocks):
            block = DenseBlock(inner_channels, growth_rate, num_layers)
            self.features.add_module(f"dense_block_{i+1}", block)
            inner_channels += num_layers * growth_rate

            # transition layer except after last one
            if i != len(num_blocks) - 1: 
                out_channels = inner_channels // 2
                trans = Transition(inner_channels, out_channels)
                self.features.add_module(f"transition_{i+1}", trans)
                inner_channels = out_channels

        # final batch norm and ReLU before pooling
        self.features.add_module("bn_final", nn.BatchNorm2d(inner_channels))
        self.features.add_module("relu_final", nn.ReLU(inplace=True))

        # final pooling and classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(inner_channels, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

## Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = DenseNet121(num_classes=4).to(device)

train_loader = dataloaders['train']
val_loader = dataloaders['valid']

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 10

# Initialize tracking lists
train_losses = []
val_losses = []

print("Starting DenseNet-121 Training...")

# Training Loop
for epoch in range(num_epochs):
    # Training Phase
    model.train()
    running_train_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
        
    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation Phase
    model.eval()
    running_val_loss = 0.0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = loss_function(outputs, labels)
            running_val_loss += loss.item()
            
    avg_val_loss = running_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

## Evaluation

In [ ]:
test_loader = dataloaders['test']

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)

        # Get predictions
        _, preds = torch.max(outputs, 1)

        # Move back to CPU and convert to numpy arrays for sklearn
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
conf_matrix = confusion_matrix(all_labels, all_preds)

print(f"Accuracy: {accuracy:.4f}")
print(f"Mean Precision: {precision:.4f}")
print(f"Mean Recall: {recall:.4f}")
print(f"Mean F1-Score: {f1:.4f}")

plt.figure(figsize=(10, 5))
epochs_range = range(1, num_epochs + 1)
plt.plot(epochs_range, train_losses, label='Training Loss', marker='o')
plt.plot(epochs_range, val_losses, label='Validation Loss', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('DenseNet-121 Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('DenseNet-121 Confusion Matrix')
plt.show()

## Reference
https://arxiv.org/pdf/1608.06993